# Блокнот 12. PyTorch: от тензора до обученной сети

Курс «Машинное обучение с нуля», модуль 12. Разделы: 1 — урок 12.1, 2 — урок 12.2, 3 — урок 12.3, 4 — урок 12.4, 5–6 — урок 12.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

В Colab включите видеокарту: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без неё всё тоже работает, только обучение медленнее.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")  # папка для скачанных наборов
print("PyTorch", torch.__version__, "| устройство:", device)

## 1. Тензоры и видеокарта (урок 12.1)

In [ ]:
a = torch.arange(6.0).reshape(2, 3)
b = torch.tensor([10.0, 20.0, 30.0])
print(a, a.shape, a.dtype)
print("a + b (транслирование):\n", a + b)
print("сумма по оси 0:", a.sum(dim=0), "| по оси 1:", a.sum(dim=1))

arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
arr[0] = 100
print("общая память с NumPy:", t, t.dtype, "→ после .float():", t.float().dtype)

**Задание 1.** Батч из 32 цветных картинок 224 × 224 — тензор формы (32, 3, 224, 224). Создайте его из нулей, а затем превратите в форму (32, 150528) через `reshape` с `-1`.

In [ ]:
batch = ...  # ваш код
flat = ...   # ваш код
assert batch.numel() == 4_816_896 and flat.shape == (32, 150528)
print("элементов:", batch.numel(), "| МБ во float32:", batch.numel() * 4 / 2 ** 20)

Сравним скорость умножения матриц на процессоре и на видеокарте (если она есть).

In [ ]:
x = torch.randn(2000, 2000)
t0 = time.time()
for _ in range(5):
    x @ x
print(f"процессор: {(time.time() - t0) / 5 * 1000:.0f} мс на умножение")
if device == "cuda":
    xg = x.to(device)
    xg @ xg
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(5):
        xg @ xg
    torch.cuda.synchronize()
    print(f"видеокарта: {(time.time() - t0) / 5 * 1000:.1f} мс на умножение")

## 2. Autograd против ручного обратного прохода (урок 12.2)

Сеть урока 10.4: $x = (1; 2)$, $z = W_1 x$, $h = \text{ReLU}(z)$, $\hat{y} = w_2 \cdot h$, $L = \frac{1}{2}(\hat{y} - 3)^2$. Руками мы получили $\partial L / \partial W_1 = [[0, 0], [-1, -2]]$ и $\partial L / \partial w_2 = (0; -4)$.

**Задание 2.** Запишите прямой проход и вызовите `backward()`.

In [ ]:
x = torch.tensor([1.0, 2.0])
W1 = torch.tensor([[0.5, -1.0], [1.0, 0.5]], requires_grad=True)
w2 = torch.tensor([1.0, 0.5], requires_grad=True)
L = ...  # ваш код: прямой проход
# ваш код: обратный проход
print("L =", L.item(), "\nW1.grad =", W1.grad, "\nw2.grad =", w2.grad)
assert torch.allclose(W1.grad, torch.tensor([[0.0, 0.0], [-1.0, -2.0]]))
assert torch.allclose(w2.grad, torch.tensor([0.0, -4.0]))

Градиенты накапливаются: вызовем прямой и обратный проход ещё раз, не обнуляя.

In [ ]:
L = 0.5 * (w2 @ torch.relu(W1 @ x) - 3) ** 2
L.backward()
print("w2.grad после второго backward:", w2.grad)  # (0; −8)
w2.grad.zero_()
W1.grad.zero_()
print("после zero_():", w2.grad)

Своя проверка — `torch.autograd.gradcheck` сравнивает autograd с численной производной (нужен float64).

In [ ]:
f = lambda w: (torch.sin(w) * w ** 2).sum()
w = torch.randn(5, dtype=torch.float64, requires_grad=True)
print("gradcheck:", torch.autograd.gradcheck(f, (w,)))

## 3. Модель: nn.Sequential и свой nn.Module (урок 12.3)

**Задание 3.** Допишите класс `MLP` с той же архитектурой, что у `seq`: Flatten → Linear(784, 256) → ReLU → Linear(256, 128) → ReLU → Linear(128, 10).

In [ ]:
seq = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        ...  # ваш код: слои

    def forward(self, x):
        ...  # ваш код: прямой проход


def count_params(model):
    return sum(p.numel() for p in model.parameters())


model = MLP()
print(model)
for name, p in model.named_parameters():
    print(f"{name:12s} {tuple(p.shape)}")
print("параметров:", count_params(model), "и", count_params(seq))
assert count_params(model) == count_params(seq) == 235146
assert model(torch.randn(8, 1, 28, 28)).shape == (8, 10)

## 4. Данные: Dataset и DataLoader (урок 12.4)

Скачиваем Fashion-MNIST (около 30 МБ). **Задание 4.** Посчитайте среднее и стандартное отклонение яркости по **обучающим** картинкам (в долях от 0 до 1) — их возьмёт `Normalize`.

In [ ]:
raw = datasets.FashionMNIST(DATA, train=True, download=True)
pixels = raw.data.float() / 255           # форма (60000, 28, 28)
mean, std = ..., ...  # ваш код
print(f"среднее {mean:.4f}, стандартное отклонение {std:.4f}")
assert abs(mean - 0.2860) < 1e-3 and abs(std - 0.3530) < 1e-3

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((mean,), (std,))])
train = datasets.FashionMNIST(DATA, train=True, download=True, transform=tf)
test = datasets.FashionMNIST(DATA, train=False, download=True, transform=tf)
CLASSES = ["футболка", "брюки", "пуловер", "платье", "пальто", "сандалия", "рубашка", "кроссовок", "сумка", "ботинок"]
train_loader = DataLoader(train, batch_size=128, shuffle=True)
test_loader = DataLoader(test, batch_size=1000)
X, y = next(iter(train_loader))
print("батч:", X.shape, y.shape, "| батчей за эпоху:", len(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, X, y):
    ax.imshow(img[0], cmap="gray_r")
    ax.set_title(CLASSES[lab], fontsize=9)
    ax.axis("off")
plt.show()

Свой `Dataset` — два метода. Пример: набор из тензоров с метками, где картинка ещё и отражается по горизонтали с вероятностью 1/2 (аугментация).

In [ ]:
class FlipDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i]
        if torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[-1])     # отражение по ширине
        return x, self.y[i]


small = FlipDataset(X, y)
print(len(small), small[0][0].shape)

## 5. Цикл обучения (урок 12.5)

**Задание 5.** Допишите пять строк итерации: обнулить градиенты, прямой проход, потеря, обратный проход, шаг.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total, n = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        loss = ...  # ваш код: пять строк итерации
        total += loss.item() * len(y)
        n += len(y)
    return total / n


def evaluate(model, loader):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(dim=1) == y).sum().item()
            n += len(y)
    return correct / n

In [ ]:
EPOCHS = 10 if device == "cuda" else 3
torch.manual_seed(0)
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
print("потеря на старте:", loss_fn(model(X.to(device)), y.to(device)).item(), "≈ ln 10 =", np.log(10))
history = []
for epoch in range(EPOCHS):
    t0 = time.time()
    loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    acc = evaluate(model, test_loader)
    history.append(acc)
    print(f"эпоха {epoch + 1}: потеря {loss:.4f}, точность на тесте {acc:.4f} ({time.time() - t0:.1f} с)")
assert history[-1] > 0.85

Сохраняем веса и загружаем их в новую модель — точность должна совпасть.

In [ ]:
torch.save(model.state_dict(), "mlp_fashion.pt")
restored = MLP()
restored.load_state_dict(torch.load("mlp_fashion.pt", map_location="cpu"))
restored = restored.to(device)
print("после загрузки:", evaluate(restored, test_loader))
assert abs(evaluate(restored, test_loader) - history[-1]) < 1e-9

## 6. Где ошибается сеть

Точность по классам и примеры ошибок. Чаще всего путаются рубашка, футболка, пуловер и пальто — у картинок 28 × 28 похожие силуэты.

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds.append(model(Xb.to(device)).argmax(1).cpu())
        labels.append(yb)
preds, labels = torch.cat(preds), torch.cat(labels)
for c, name in enumerate(CLASSES):
    mask = labels == c
    print(f"{name:10s} {(preds[mask] == c).float().mean():.3f}")

wrong = torch.where(preds != labels)[0][:8]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, wrong):
    ax.imshow(test[i][0][0], cmap="gray_r")
    ax.set_title(f"{CLASSES[labels[i]]}\n→ {CLASSES[preds[i]]}", fontsize=8)
    ax.axis("off")
plt.show()

Попробуйте сами: добавьте `nn.Dropout(0.2)` после скрытых слоёв, поменяйте шаг или число нейронов и сравните точность. В модуле 13 свёрточная сеть на тех же данных обойдёт этот MLP.